# Early warning for students at risk of failing: analysis

**Authors:** Mohamed Reda Zarrouk and Georgy Salameh. PGE 2 final project (Machine Learning ×
Advanced Python).

This notebook tells the modelling story. **All the logic lives in the `student_risk` package.**
The notebook only calls it, and reads the result tables that `student-risk train` wrote in
`reports/`.

**How to run it:**

```bash
pip install -e ".[notebook]"
jupyter nbconvert --to notebook --execute --inplace notebooks/analysis.ipynb
```

If no trained model exists yet, the first code cell runs the package's `train` step. That step
is deterministic (seed 42), so it recreates exactly the same model and reports.

**Rule followed everywhere:** every choice (model, hyperparameters, threshold) was made on the
**training set** with cross-validation. The test set is used **once**, in section 8, to report
the final score.

In [1]:
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import cross_validate

from student_risk.cli import run_train
from student_risk.config import load_config
from student_risk.data import DatasetLoader
from student_risk.evaluate import (
    SCORING,
    coefficient_table,
    final_scores,
    group_report,
    wilson_interval,
)
from student_risk.features import build_preprocessor
from student_risk.models import Trainer, make_cv
from student_risk.plots import plot_coefficients, plot_confusion_matrix, plot_precision_recall

pd.set_option("display.precision", 3)
config = load_config()
figures_dir = config.reports_dir / "figures"

if not config.model_path.exists():  # fresh clone: train once with the package
    run_train(config)
trainer = Trainer.load(config.model_path, config)
print(f"Loaded model: {trainer.best_name}, threshold {trainer.threshold:.3f}")

Loaded model: logreg_l2, threshold 0.140


## 1. Problem framing

- **Who uses the tool:** a secondary school's teaching team, at the **start of the year**.
- **Decision it supports:** which students to offer tutoring, *before* grades are known.
- **Target:** `fail = 1` if the final grade G3 < 10 (out of 20), else `0`. This is binary
  classification, and "fail" is the positive, minority class.
- **Cost of each mistake:**
  - **False negative** (a failing student not flagged): the student gets no help and may fail
    the year. This is the **costly** error.
  - **False positive** (a passing student flagged): some tutoring time is spent on a student who
    would have passed. This is cheap, and extra help is rarely harmful.
- **Consequence:** we favour **recall**, and require recall ≥ 0.80 (from `config.yaml`). Among
  the thresholds that satisfy it, we choose the one with the best precision, so that tutors are
  not flooded with false alarms.

## 2. Data exploration

We use the UCI *Student Performance* dataset (id 320, CC BY 4.0), loaded with `ucimlrepo` by
`DatasetLoader`.

In [2]:
loader = DatasetLoader(config)
raw = loader.load_raw()
print(f"Raw data: {raw.shape[0]} rows x {raw.shape[1]} columns (30 features + G1, G2, G3)")
print("Column types:", raw.dtypes.astype(str).value_counts().to_dict())
print("Missing values:", int(raw.isna().sum().sum()))
print("Duplicate rows:", int(raw.duplicated().sum()))

Raw data: 649 rows x 33 columns (30 features + G1, G2, G3)
Column types: {'object': 17, 'int64': 16}
Missing values: 0
Duplicate rows: 0


In [3]:
df = loader.clean(raw)
X, y = loader.build_target(df)
print(f"After cleaning: {len(df)} students")
print(f"Fail rate: {y.mean():.3f} ({y.sum()} fails, {len(y) - y.sum()} passes)")
print(
    "Numeric features:",
    X.select_dtypes(include="number").shape[1],
    "| text (categorical) features:",
    X.select_dtypes(exclude="number").shape[1],
)

After cleaning: 649 students
Fail rate: 0.154 (100 fails, 549 passes)
Numeric features: 13 | text (categorical) features: 17


**Class balance:** about 15 % of students fail. A model that always says "pass" would be about
85 % accurate and useless, so accuracy is never used. We use recall, precision, F1, ROC-AUC and
average precision.

### Special code: G3 = 0

In [4]:
zero = df["G3"] == 0
print(f"Students with G3 = 0: {zero.sum()}, of whom {(zero & (df['G2'] > 0)).sum()} had G2 > 0")

Students with G3 = 0: 15, of whom 8 had G2 > 0


A final grade of exactly 0 after a non-zero period-2 grade suggests the student did **not sit
the final exam**, rather than answering everything wrong **[from prior knowledge, to verify]**.

**Our decision:** keep them as `fail = 1`. For the school, a student who ends the year with 0 did
not pass, and missing the exam is itself a sign of risk. Section 10 checks how much this choice
matters, using cross-validation only.

### Leakage: why G1 and G2 are dropped

In [5]:
print("Correlation with G3:", df[["G1", "G2", "G3"]].corr()["G3"].round(3).to_dict())
print(
    "Feature columns kept:",
    X.shape[1],
    "| G1, G2 or G3 among them:",
    sorted({"G1", "G2", "G3"} & set(X.columns)),
)

Correlation with G3: {'G1': 0.826, 'G2': 0.919, 'G3': 1.0}
Feature columns kept: 30 | G1, G2 or G3 among them: []


G1 and G2 (the period 1 and period 2 grades) are almost the final grade: correlation 0.83 and
0.92 with G3. A model using them would mostly copy G2. More importantly, **they are not known at
the start of the year**, when the early warning is needed. So they are dropped, and G3 is
dropped after building the target.

**The split:** one stratified train/test split with seed 42. 20 % of the students are kept aside.

In [6]:
X_train, X_test, y_train, y_test = loader.split(X, y)
print(
    f"Train: {len(y_train)} students ({y_train.sum()} fails) | "
    f"Test: {len(y_test)} students ({y_test.sum()} fails)"
)
print("No student in both sets:", set(X_train.index).isdisjoint(X_test.index))

Train: 519 students (80 fails) | Test: 130 students (20 fails)
No student in both sets: True


## 3. Pipeline

All preparation is inside one scikit-learn `Pipeline`. Its first step is the
`ColumnTransformer` built by `features.build_preprocessor()`:

- numeric columns: median imputation, then standardisation;
- text columns: most-frequent imputation, then one-hot encoding (unknown categories ignored).

Because the whole pipeline is refitted on each training fold, the scaler and the encoder never
see the validation fold. This is what prevents leakage in cross-validation.

In [7]:
for name, steps, _columns in build_preprocessor().transformers:
    print(
        f"{name:12s} ->", " -> ".join(f"{step}: {type(obj).__name__}" for step, obj in steps.steps)
    )

numeric      -> impute: SimpleImputer -> scale: StandardScaler
categorical  -> impute: SimpleImputer -> onehot: OneHotEncoder


## 4. Baseline

Results from `reports/baseline.csv`, written by `student-risk train` (5-fold stratified CV on the
training set; recall, precision and F1 at threshold 0.5):

In [8]:
baseline = pd.read_csv(config.reports_dir / "baseline.csv", index_col=0)
baseline

,roc_auc,average_precision,recall,precision,f1
dummy_most_frequent,0.500,0.154,0.0,0.000,0.00
logistic_regression,0.779,0.457,0.3,0.503,0.37


The Dummy (always "pass") catches nobody: recall 0, and an average precision equal to the fail
rate. An untuned logistic regression ranks students much better (ROC-AUC 0.78, AP 0.46). But at
the default threshold of 0.5 it catches only 30 % of failing students.

## 5. Model comparison

Four models, each tuned with `GridSearchCV` (grids in `config.yaml`) under **the same**
5-fold `StratifiedKFold`, and refitted on **average precision**. AP suits a rare positive class
and does not depend on a threshold. Results are from `reports/model_comparison.csv`.

In [9]:
comparison = pd.read_csv(config.reports_dir / "model_comparison.csv", index_col=0)
columns = [
    "average_precision_mean",
    "average_precision_std",
    "roc_auc_mean",
    "train_average_precision_mean",
    "recall_mean",
    "f1_mean",
]
display(comparison[columns])
comparison[["best_params"]]

,average_precision_mean,average_precision_std,roc_auc_mean,train_average_precision_mean,recall_mean,f1_mean
logreg_l2,0.500,0.143,0.803,0.560,0.050,0.091
logreg_l1,0.538,0.158,0.828,0.564,0.237,0.331
linear_svm,0.500,0.147,0.803,0.552,0.225,0.310
knn,0.510,0.179,0.768,1.000,0.050,0.090


,best_params
logreg_l2,"{'clf__C': 0.01, 'clf__class_weight': None}"
logreg_l1,"{'clf__C': 0.1, 'clf__class_weight': None}"
linear_svm,"{'clf__estimator__C': 0.001, 'clf__estimator__..."
knn,"{'clf__n_neighbors': 31, 'clf__weights': 'dist..."


## 6. Selection rule

We take the model with the highest mean CV AP. **But** if a simpler model is within one standard
deviation of it, we prefer the simpler one. The simplicity order is: L2 logistic regression,
L1 logistic regression, linear SVM, k-NN.

- `logreg_l1` has the highest mean AP (0.538), with a std of 0.158.
- All four models are within 0.158 of it, so the rule picks the simplest: **`logreg_l2`**
  (C = 0.01, AP 0.500).
- k-NN overfits: its train AP is 1.000 against 0.510 in CV.
- The CV score of the chosen model is slightly **optimistic**, because hyperparameters and model
  were chosen on the same folds (selection bias). Section 8 gives the honest estimate.

In [10]:
print("Model chosen by the package:", trainer.best_name)
print("Parameters:", comparison.loc[trainer.best_name, "best_params"])

Model chosen by the package: logreg_l2
Parameters: {'clf__C': 0.01, 'clf__class_weight': None}


## 7. Threshold on out-of-fold predictions

`Trainer.tune_threshold` computes **out-of-fold** probabilities for the training students (each
student is predicted by a model that did not see them, on the same 5 folds). It then chooses the
threshold with recall ≥ 0.80 and the best precision.

Below, we recompute them with the package and check that we get back exactly the saved threshold
and the saved reports.

In [11]:
saved_threshold = trainer.threshold
trainer.tune_threshold(X_train, y_train)
assert trainer.threshold == saved_threshold, "threshold differs from the saved model"

threshold_table = trainer.threshold_comparison(y_train)
tradeoff = trainer.threshold_tradeoff(y_train)
pd.testing.assert_frame_equal(
    threshold_table.astype(float),
    pd.read_csv(config.reports_dir / "threshold_report.csv", index_col=0).astype(float),
)
pd.testing.assert_frame_equal(tradeoff, pd.read_csv(config.reports_dir / "threshold_tradeoff.csv"))
print(f"Threshold {trainer.threshold:.4f}: identical to the saved model and to reports/")
threshold_table

Threshold 0.1403: identical to the saved model and to reports/


,threshold,precision,recall,f1,flagged_share,tn,fp,fn,tp
default_0.5,0.50,0.571,0.05,0.092,0.013,436.0,3.0,76.0,4.0
chosen,0.14,0.312,0.80,0.449,0.395,298.0,141.0,16.0,64.0


In [12]:
plot_precision_recall(
    y_train, trainer.oof_proba, trainer.threshold, figures_dir / "precision_recall_oof.png"
)
tradeoff

,min_recall,threshold,precision,recall,flagged_share
0,0.7,0.171,0.389,0.725,0.287
1,0.8,0.140,0.312,0.800,0.395
2,0.9,0.116,0.248,0.900,0.559


![Precision-recall curve](../reports/figures/precision_recall_oof.png)

**What the tables say:**

- At the default threshold of 0.5, the model catches only 4 of the 80 failing students. Its
  probabilities are squeezed low by the strong regularisation (C = 0.01).
- At the chosen threshold of **0.140**, it catches 64 of 80 (recall 0.80) with precision 0.31.
  This means flagging 39.5 % of students.
- **The trade-off table shows the price of recall:**
  - recall 0.70 → 28.7 % of students flagged;
  - recall 0.80 → 39.5 %;
  - recall 0.90 → 55.9 %.

  The school can pick the row that matches its tutoring capacity.

## 8. Single test evaluation

The model and threshold were fixed above, without looking at the test set. We now score them
**once** on the 130 test students. Nothing is changed afterwards.

In [13]:
test_proba = trainer.best_model.predict_proba(X_test)[:, 1]
scores = final_scores(y_test, test_proba, trainer.threshold)
plot_confusion_matrix(scores, figures_dir / "confusion_matrix_test.png")
pd.Series(scores).round(3)

roc_auc               0.778
average_precision     0.346
threshold             0.140
precision             0.300
recall                0.750
f1                    0.429
flagged_share         0.385
tn                   75.000
fp                   35.000
fn                    5.000
tp                   15.000
dtype: float64

In [14]:
caught, n_fail = scores["tp"], scores["tp"] + scores["fn"]
low, high = wilson_interval(caught, n_fail)
print(
    f"Test recall = {caught}/{n_fail} = {caught / n_fail:.3f}, "
    f"95 % Wilson interval [{low:.3f}, {high:.3f}]"
)

Test recall = 15/20 = 0.750, 95 % Wilson interval [0.531, 0.888]


![Test confusion matrix](../reports/figures/confusion_matrix_test.png)

**Reading the test result:**

- ROC-AUC 0.778, AP 0.346.
- At threshold 0.140, the model catches **15 of the 20** failing test students (recall 0.75),
  with precision 0.30, and it flags 38.5 % of students. It misses **5** failing students and
  raises 35 false alarms.
- **The recall is just below the 0.80 target,** but the test set is small. The **95 % Wilson
  interval for recall is [0.53, 0.89]**: with only 20 failing students, the data cannot tell a
  true recall of 0.75 from one of 0.80.
- **Why Wilson:** it stays inside [0, 1] and behaves well for small samples and proportions near
  0 or 1, where the simple "±1.96·√(p(1−p)/n)" interval is unreliable **[from prior knowledge,
  to verify]**.
- **The AP drop** (0.500 in CV to 0.346 on test) agrees with the selection-bias optimism and with
  the small test set.

## 9. Interpretation: coefficients of the L2 logistic regression

**How to read a coefficient:**

- It is the change in the **log-odds of failing**.
- **Numeric features are standardised**, so a coefficient is the effect of **+1 standard
  deviation** of that feature (the std of the training set is shown below). `odds_ratio` =
  e^coefficient is the factor by which the odds of failing are multiplied.
- **Text features are one-hot encoded with every category kept.** With L2, the two columns of a
  yes/no feature get opposite coefficients, for example `school_GP` −0.173 and `school_MS`
  +0.173. The meaningful quantity is the **difference** between categories: MS rather than GP
  adds 0.346 to the log-odds.
- Coefficients are small overall, because C = 0.01 is a strong penalty.
- **They are associations, not causes.**

In [15]:
coefficients = coefficient_table(trainer.best_model)
scaler = trainer.best_model.named_steps["preprocess"].named_transformers_["numeric"]
std = pd.Series(scaler.named_steps["scale"].scale_, index=trainer.numeric_columns)
coefficients["train_std (numeric only)"] = coefficients["feature"].map(std)
plot_coefficients(coefficients, figures_dir / "coefficients.png", top=15)
coefficients.head(15)

,feature,coefficient,odds_ratio,train_std (numeric only)
0,failures,0.340,1.404,0.632
1,school_GP,-0.173,0.841,NaN
2,school_MS,0.173,1.189,NaN
3,studytime,-0.167,0.846,0.813
4,Fedu,-0.109,0.897,1.087
5,higher_yes,-0.096,0.909,NaN
6,higher_no,0.096,1.100,NaN
7,freetime,0.080,1.083,1.057
8,Walc,0.073,1.075,1.275
9,Medu,-0.071,0.931,1.122


![Top coefficients](../reports/figures/coefficients.png)

**Top features and the meaning of their signs** (training set, chosen model):

- **`failures` (+0.340, the largest):** each extra standard deviation of past class failures
  (std ≈ 0.6 failures) multiplies the odds of failing by about 1.4. **Past failure is the
  strongest warning sign.**
- **`school` (MS +0.173 / GP −0.173):** students of the MS school have higher predicted risk.
  In the training data, 31 % of MS students fail against 7 % of GP students.
- **`studytime` (−0.167):** more weekly study time lowers the risk.
- **`Fedu` (−0.109) and `Medu` (−0.071):** higher parental education is associated with a lower
  risk.
- **`higher` (no +0.096 / yes −0.096):** students who do not want to go on to higher education
  are at higher risk.
- **`freetime` (+0.080), `Walc` (+0.073, weekend alcohol), `absences` (+0.058) and `goout`
  (+0.049):** more free time, alcohol, absences and outings go with a higher risk.

## 10. Sensitivity check: the G3 = 0 students (training set only)

Same model and same hyperparameters (a clone of the saved pipeline), evaluated with the same
5-fold CV on the **training set**, with and without the students whose G3 = 0. **This is a
report only: nothing is changed.**

In [16]:
g3_train = df.loc[X_train.index, "G3"]
keep = (g3_train != 0).to_numpy()
rows = {}
for name, (X_part, y_part) in {
    "with G3 = 0 (as trained)": (X_train, y_train),
    "without G3 = 0": (X_train[keep], y_train[keep]),
}.items():
    cv = cross_validate(
        clone(trainer.best_model), X_part, y_part, cv=make_cv(config), scoring=SCORING
    )
    rows[name] = {
        "students": len(y_part),
        "fails": int(y_part.sum()),
        "fail_rate": y_part.mean(),
        "roc_auc_mean": cv["test_roc_auc"].mean(),
        "average_precision_mean": cv["test_average_precision"].mean(),
        "average_precision_std": cv["test_average_precision"].std(),
    }
sensitivity = pd.DataFrame(rows).T
display(sensitivity)

zero_train = (g3_train == 0).to_numpy()
flagged = trainer.oof_proba[zero_train] >= trainer.threshold
print(
    f"G3 = 0 training students flagged (out-of-fold, threshold {trainer.threshold:.3f}): "
    f"{flagged.sum()} of {zero_train.sum()}"
)

,students,fails,fail_rate,roc_auc_mean,average_precision_mean,average_precision_std
with G3 = 0 (as trained),519.0,80.0,0.154,0.803,0.500,0.143
without G3 = 0,508.0,69.0,0.136,0.805,0.464,0.114


G3 = 0 training students flagged (out-of-fold, threshold 0.140): 9 of 11


**Reading it:**

- The 11 training students with G3 = 0 are removed, leaving 69 fails out of 508.
- ROC-AUC barely moves (0.803 → 0.805).
- AP drops from 0.500 to 0.464, but **AP depends on the fail rate** (15.4 % → 13.6 %), so the two
  APs are not directly comparable. Both are well within one standard deviation of each other.
- The G3 = 0 students are **not** hard cases: 9 of the 11 are flagged out-of-fold.
- **Conclusion:** keeping them as fails does not distort the model, and they are students the
  school would want flagged. Our decision stands, unchanged.

## 11. Fairness check: recall and flagged rate by group (training set, out-of-fold)

`group_report` applies the chosen threshold to the out-of-fold probabilities, separately for
each group. **Report only.** The 95 % Wilson interval on each group's recall shows how uncertain
these numbers are.

In [17]:
def with_recall_interval(table):
    """Add the 95 % Wilson interval of each group's recall to a group_report table."""
    bounds = [
        wilson_interval(round(r * f), int(f))
        for r, f in zip(table["recall"], table["n_fail"], strict=True)
    ]
    table["recall_95%_low"] = [low for low, _ in bounds]
    table["recall_95%_high"] = [high for _, high in bounds]
    return table


for column in ["sex", "school"]:
    print(f"By {column}:")
    display(
        with_recall_interval(
            group_report(y_train, trainer.oof_proba, trainer.threshold, X_train[column])
        )
    )

By sex:


,n_students,n_fail,recall,precision,flagged_share,recall_95%_low,recall_95%_high
F,303.0,40.0,0.75,0.283,0.350,0.598,0.858
M,216.0,40.0,0.85,0.343,0.458,0.709,0.929


By school:


,n_students,n_fail,recall,precision,flagged_share,recall_95%_low,recall_95%_high
GP,339.0,25.0,0.680,0.215,0.233,0.484,0.828
MS,180.0,55.0,0.855,0.373,0.700,0.738,0.924


**Reading it:**

- **By sex:**
  - recall is 0.75 for girls (F) and 0.85 for boys (M);
  - boys are flagged more often (45.8 % against 35.0 %).
- **By school:**
  - GP: recall 0.68, 23.3 % flagged. MS: recall 0.86, **70 % flagged**.
  - The model uses the school as a strong signal (section 9). As a result, MS students are
    flagged much more often, and failing GP students are missed more often.
- **⚠ Small groups:** each recall is computed on only 25 to 55 failing students, so the Wilson
  intervals are wide and overlap. These differences are **warnings to investigate, not proven
  biases**. They come from the training set (out-of-fold), not from new data.
- **What this means for a school:**
  - A failing student at GP is less likely to be caught.
  - An MS student is very likely to be flagged whatever their own profile.
  - A school using the tool should monitor recall per group, and might consider per-school
    thresholds. That would be a **new modelling decision, which we did not make**.

## 12. Two limits of the model

1. **Small data and optimistic selection.** With 519 training students (80 fails), the CV scores
   vary a lot from fold to fold (AP std ≈ 0.14). All four models are statistically tied.
   - Choosing the best on the same folds makes the CV score optimistic: AP 0.500 in CV against
     0.346 on test.
   - The test set itself has only 20 fails, so test recall is uncertain (95 % interval
     [0.53, 0.89]).
2. **Limited signal without grades.**
   - Once G1 and G2 are excluded, the profile features (past failures, school, study time,
     family background) carry only moderate information.
   - Reaching recall 0.80 requires flagging about 40 % of all students, with only about 1 in 3
     flagged students actually failing.
   - The model is linear: it cannot capture interactions between features unless they are
     added by hand.
   - The data come from two Portuguese schools in one period. It may not transfer to other
     schools, countries or years **[from prior knowledge, to verify: the context of the data]**.

## 13. Risks of using it on real students

**Errors** affect real people:

- About 1 in 4 failing students is missed (5 of 20 in the test set). A school must not treat
  "not at risk" as a guarantee.
- About 2 in 3 flagged students would have passed. A flag must lead to an *offer* of support, and
  never to a sanction, a lower expectation, or a label in the student's file.

**Fairness:**

- The model leans on the school attended, past failures and parental education. These reflect
  social background as much as individual effort.
- The rates differ between groups (section 11), so the tool could reinforce existing
  inequalities.
- A school would need to check outcomes per group regularly, and keep a human decision-maker
  in the loop.

**Privacy:**

- The inputs include sensitive personal data: family situation, parents' jobs and education,
  alcohol consumption, health, romantic relationships.
- Using them requires a clear legal basis and informed consent (for example, the GDPR in
  Europe), strict access control, keeping data only as long as needed, and removing features
  that are not necessary **[from prior knowledge, to verify: the applicable legal framework]**.
- Students and families should be told the tool exists, and be able to question a decision.